In [1]:
import os
os.chdir(r"D:\Study\Programs\trading")

In [2]:
from strategies_dev.utils import resample_fractional_minute, getAggregatedVolume
from pathlib import Path
import pandas as pd
import numpy as np

In [3]:
date_ = "30APR2026"
file_names_root = ["NIFTY2650523950CE", "NIFTY2650523950PE", "NIFTY2650523900CE", "NIFTY2650523900PE"]
file_names = [file_name + ".xlsx" for file_name in file_names_root]
file_names_tns = [file_name + "_tns.xlsx" for file_name in file_names_root]

N_list = [1, 4, 5, 6, 7, 8, 9, 10, 12]

In [4]:
def get_colname(file_name, df):
    if "PE" in file_name or "CE" in file_name:
        col_name = "last_trade_time"
        df["volume_at_tick"] = df["volume_traded"].diff()
        df.loc[df["volume_at_tick"] == 0, "volume_at_tick"] = np.nan
    else:
        col_name = "local_time"
    return  col_name

In [5]:
def get_clubbed(price_df, tns_df, col_name, N_list):
    price_df[col_name] = pd.to_datetime(price_df[col_name])
    for N in N_list:
        clubbed_data = resample_fractional_minute(price_df, col_name, n=N, depth=False)
        clubbed_data["bucket_time"] = pd.to_datetime(clubbed_data["bucket_time"])
        clubbed_data["date_str"] = clubbed_data["bucket_time"].dt.strftime("%d%b%Y").str.upper()
        clubbed_data = clubbed_data[clubbed_data["date_str"] == date_]
        clubbed_data = clubbed_data.drop(columns=["date_str"])
        clubbed_data["date"] = clubbed_data["bucket_time"]

        #tns
        clubbed_tns = getAggregatedVolume(tns_df, N)
        # 1. Convert the string column to datetime
        clubbed_tns['bucket_timestamp'] = pd.to_datetime(clubbed_tns['bucket_timestamp'])
        # 2. Now perform the merge
        clubbed_data = clubbed_data.merge(
            clubbed_tns,
            how="left",
            left_on="bucket_time",
            right_on="bucket_timestamp"
        )
        clubbed_data.rename(columns={'quantity': 'actual_volume'}, inplace=True)

        # Join actual volume information
        folder = Path(fr"D:\Study\Programs\trading\assets\logs\{date_}\candles")
        folder.mkdir(parents=True, exist_ok=True)
        file_path = os.path.join(folder, file_name.split(".")[0] + f"_N={N}.xlsx")
        clubbed_data.to_excel(file_path, index=False)

In [6]:
def get_tns(tns_df):
    tns_df["date"] = date_
    combined_dt = tns_df['date'].astype(str) + ' ' + tns_df['time'].astype(str)
    tns_df['timestamp'] = pd.to_datetime(combined_dt)
    tns_df['timestamp'] = tns_df['timestamp'].dt.strftime('%d-%m-%Y %H:%M:%S')
    return tns_df

In [7]:
for file_name, file_name_tns in zip(file_names, file_names_tns):
    file_path = Path(fr"D:\Study\Programs\trading\assets\logs\{date_}\extracted_symbols\{file_name}")
    tns_file_path = Path(fr"D:\Study\Programs\trading\assets\logs\{date_}\tns\{file_name_tns}")
    price_df = pd.read_excel(file_path)
    tns_df = pd.read_excel(tns_file_path)
    tns_df = get_tns(tns_df)
    col_name = get_colname(file_name, price_df)
    get_clubbed(price_df, tns_df, col_name, N_list)

C:\Users\bhatt\AppData\Local\Temp\ipykernel_122460\105448797.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  tns_df['timestamp'] = pd.to_datetime(combined_dt)
C:\Users\bhatt\AppData\Local\Temp\ipykernel_122460\1980372139.py:14: UserWarning: Parsing dates in %d-%m-%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  clubbed_tns['bucket_timestamp'] = pd.to_datetime(clubbed_tns['bucket_timestamp'])
C:\Users\bhatt\AppData\Local\Temp\ipykernel_122460\1980372139.py:14: UserWarning: Parsing dates in %d-%m-%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  clubbed_tns['bucket_timestamp'] = pd.to_datetime(clubbed_tns['bucket_timestamp'])
C:\Users\bhatt\AppData\Local\Temp\ipykernel_122460\

In [48]:
# clubbed_price

In [49]:
# clubbed_price.head()